# Adım 7: Literatür Ajanı (RAG'ın tamamlanması)

İlk ajan. Konseye giden bir vaka için bir **kanıt paketi** hazırlanır ve bir dil modeline (LLM) verilir; model, kaynak gösteren tarafsız bir **kanıt özeti** yazar. Retrieval (getirme) Adım 4 ve 6'da kurulmuştu; bu adımda Augmented Generation (zenginleştirilmiş üretim) eklenerek RAG tamamlanır.

**Kanıt paketinde ne var?** (Ajan görüntüyü görmez, yalnızca metin alır.)
- **[HASTA]** yaş, cinsiyet, lezyon bölgesi
- **[CNN]** klinik eşikli karar, kalibre güven ve ilk 3 olasılık
- **[KAPI]** konsey gerekliliği puanı ve ateşlenen bulanık kurallar (Adım 5)
- **[ARSIV]** en benzer 50 eğitim vakasının oylaması (Adım 4)
- **[V1]…[V5]** en benzer 5 arşiv vakası: tanı, tanının nasıl doğrulandığı (patoloji, takip vb.), benzerlik
- **[M1]…[M6]** PubMed makaleleri (Adım 6). Sorgular, aday tanılardan (CNN kararı, CNN'in ve arşivin en olası sınıfı) **kuralla** üretilir; örneğin "dermoscopic criteria to differentiate melanoma from melanocytic nevus".

**Ajandan istenen (JSON):** 3-5 cümlelik özet, aday tanıların lehine ve aleyhine kanıtlar, en olası tanı, kanıt yönü (CNN'i *destekliyor / zayıflatıyor / belirsiz*) ve uzmanın bakması gereken dermoskopik bulgular. **Her iddia kaynak göstermek zorunda.**

**Güvenilirlik önlemleri:**
1. **Kaynak doğrulaması:** Cevaptaki her `[...]` etiketi pakette olmalı. Pakette olmayan bir etiket (ör. `[M9]` ya da uydurma bir PMID) **uydurma kaynak** olarak sayılır.
2. **Görüntü yok kuralı:** Ajan "bu lezyonda şu bulgu var" diyemez; literatürdeki bulguları uzmanın bakacağı ölçütler olarak yazar.
3. **Tekrarlanabilirlik:** Sıcaklık 0. Her istek ve cevap Drive'a (`ajanlar/onbellek/`) kaydedilir; aynı istek tekrar yapılmaz.
4. **Test kümesine dokunulmaz:** Bu adım **geliştirme** adımıdır. Yalnızca **doğrulama** kümesinden 20 konsey vakası kullanılır (yarısında CNN yanlış, yarısında doğru). Ajanların gerçek katkısı Adım 11'de testte ölçülecek.

**Başlamadan önce:** Colab 🔑 Secrets'ta `GEMINI_API_KEY` olmalı (Notebook access açık). Adım 4, 5 ve 6 çalıştırılmış olmalı.

**Çalışma zamanı:** CPU yeterli (T4 daha hızlı). Süre yaklaşık 10-15 dakika; çoğu dil modelinin dakikalık istek sınırını beklemekle geçer.

## 1. Ortam ve ayarlar

In [1]:
!pip -q install google-genai faiss-cpu
# Başka sağlayıcı için:  !pip -q install anthropic   ya da   !pip -q install openai
import os, sys, json, time, importlib, warnings
import numpy as np
import pandas as pd

from google.colab import drive, userdata
drive.mount('/content/drive')
PROJE = '/content/drive/MyDrive/bitirme'

# ---- Ayarlar ----
SAGLAYICI = 'gemini'          # 'gemini', 'anthropic' ya da 'openai'
MODEL = 'gemini-3.5-flash'    # kararlı ve daha az yoğun; boş bırakılırsa hesaptaki en yeni kararlı 'flash' modeli seçilir
DAKIKADA_ISTEK = 8            # ücretsiz katmanın dakikalık sınırının altında kalmak için
N_ORNEK = 20                  # geliştirme için doğrulama kümesinden kaç konsey vakası
TOHUM = 42
SORGU_MODELI, CAPRAZ_MODEL = 'ncbi/MedCPT-Query-Encoder', 'ncbi/MedCPT-Cross-Encoder'
ON = '07'
ANAHTAR_ADI = {'gemini': 'GEMINI_API_KEY', 'anthropic': 'ANTHROPIC_API_KEY', 'openai': 'OPENAI_API_KEY'}[SAGLAYICI]
try:
    API_ANAHTARI = userdata.get(ANAHTAR_ADI)
except Exception:
    raise RuntimeError(f'Colab Secrets içinde {ANAHTAR_ADI} bulunamadı (ya da Notebook access kapalı).')

rng = np.random.default_rng(TOHUM)
warnings.filterwarnings('ignore', category=FutureWarning)
for d in ['src', 'ajanlar/onbellek']:
    os.makedirs(f'{PROJE}/{d}', exist_ok=True)
for gerekli in ['src/bulanik_kapi.py', 'src/pubmed_arama.py', 'rag/pubmed.faiss', 'rag/arsiv_meta.csv', 'bulanik/05_kapi_ayarlari.json']:
    assert os.path.exists(f'{PROJE}/{gerekli}'), f'{gerekli} yok: önce Adım 4, 5 ve 6 çalıştırılmalı.'
print('Ayarlar tamam.')

Mounted at /content/drive
Ayarlar tamam.


## 2. Modüller
İki dosya Drive'daki `src/` klasörüne yazılır; sonraki ajanlar ve web arayüzü de bunları kullanacak. GitHub'daki `src/` klasörüne de konacaklar.
- `llm_istemci.py`: Sağlayıcıdan bağımsız dil modeli istemcisi (önbellek, hız sınırı, yeniden deneme).
- `literatur_ajani.py`: Kanıt paketi, sistem istemi ve kaynak doğrulaması.

In [2]:
%%writefile /content/drive/MyDrive/bitirme/src/llm_istemci.py
"""Sağlayıcıdan bağımsız dil modeli istemcisi.

- Sağlayıcılar: 'gemini' (google-genai), 'anthropic', 'openai'. Değiştirmek için yalnızca saglayici ve model değişir.
- Önbellek: Her istek (sağlayıcı, model, sıcaklık, sistem istemi, mesaj) özetiyle Drive'a kaydedilir. Aynı istek
  ikinci kez yapılırsa API'ye gidilmez; böylece sonuçlar tekrarlanabilir olur ve aynı cevaba iki kez para ödenmez.
- Hız sınırı: dakikada en fazla `dakikada_istek` istek. Kota hatalarında (429 vb.) bekleyip yeniden dener.
- Zaman aşımı: Bir istek `zaman_asimi_sn` saniyede cevap gelmezse kesilir ve yeniden denenir (sunucu yoğunken takılı kalmayı önler).
"""
import os
import re
import json
import time
import hashlib

KOTA_IZLERI = ('429', 'RESOURCE_EXHAUSTED', 'rate limit', 'Rate limit', 'quota', 'Quota', 'overloaded', '503', '529', 'UNAVAILABLE',
               '504', 'DEADLINE_EXCEEDED', 'timed out', 'Timeout', 'timeout')
KALICI_HATA_IZLERI = ('400', '401', '403', '404', 'INVALID_ARGUMENT', 'PERMISSION_DENIED', 'NOT_FOUND', 'API key not valid',
                      'UNAUTHENTICATED', 'invalid_api_key', 'model_not_found')


class LLMIstemci:
    def __init__(self, saglayici, model, api_anahtari, onbellek=None, dakikada_istek=8, sicaklik=0.0, en_fazla_token=8192,
                 zaman_asimi_sn=120):
        self.saglayici, self.model, self.sicaklik = saglayici, model, sicaklik
        self.dakikada_istek, self.en_fazla_token, self.onbellek = dakikada_istek, en_fazla_token, onbellek
        self._son = 0.0
        self.istatistik = {'api_istegi': 0, 'onbellekten': 0, 'girdi_token': 0, 'cikti_token': 0}
        if onbellek:
            os.makedirs(onbellek, exist_ok=True)
        if saglayici == 'gemini':
            from google import genai
            from google.genai import types
            self._istemci = genai.Client(api_key=api_anahtari, http_options=types.HttpOptions(timeout=int(zaman_asimi_sn * 1000)))
        elif saglayici == 'anthropic':
            import anthropic
            self._istemci = anthropic.Anthropic(api_key=api_anahtari, timeout=zaman_asimi_sn, max_retries=0)
        elif saglayici == 'openai':
            from openai import OpenAI
            self._istemci = OpenAI(api_key=api_anahtari, timeout=zaman_asimi_sn, max_retries=0)
        else:
            raise ValueError(f'Bilinmeyen sağlayıcı: {saglayici}')

    def _anahtar(self, sistem, mesaj, json_cikti):
        ham = json.dumps([self.saglayici, self.model, self.sicaklik, sistem, mesaj, json_cikti], ensure_ascii=False)
        return hashlib.sha256(ham.encode('utf-8')).hexdigest()[:32]

    def _ham_cagri(self, sistem, mesaj, json_cikti):
        """(metin, girdi_token, cikti_token) döndürür."""
        if self.saglayici == 'gemini':
            from google.genai import types
            ayar = types.GenerateContentConfig(system_instruction=sistem, temperature=self.sicaklik,
                                               max_output_tokens=self.en_fazla_token,
                                               response_mime_type='application/json' if json_cikti else None)
            r = self._istemci.models.generate_content(model=self.model, contents=mesaj, config=ayar)
            u = r.usage_metadata
            cikti = (getattr(u, 'candidates_token_count', 0) or 0) + (getattr(u, 'thoughts_token_count', 0) or 0)
            return (r.text or ''), (getattr(u, 'prompt_token_count', 0) or 0), cikti
        if self.saglayici == 'anthropic':
            r = self._istemci.messages.create(model=self.model, max_tokens=self.en_fazla_token, temperature=self.sicaklik,
                                              system=sistem, messages=[{'role': 'user', 'content': mesaj}])
            metin = ''.join(getattr(b, 'text', '') for b in r.content)
            return metin, r.usage.input_tokens, r.usage.output_tokens
        r = self._istemci.chat.completions.create(
            model=self.model, temperature=self.sicaklik,
            messages=[{'role': 'system', 'content': sistem}, {'role': 'user', 'content': mesaj}],
            **({'response_format': {'type': 'json_object'}} if json_cikti else {}))
        return (r.choices[0].message.content or ''), r.usage.prompt_tokens, r.usage.completion_tokens

    def cagir(self, sistem, mesaj, json_cikti=True, deneme_sayisi=6):
        anahtar = self._anahtar(sistem, mesaj, json_cikti)
        yol = f'{self.onbellek}/{anahtar}.json' if self.onbellek else None
        if yol and os.path.exists(yol):
            with open(yol, encoding='utf-8') as f:
                kayit = json.load(f)
            self.istatistik['onbellekten'] += 1
            return {**kayit, 'onbellekten': True}
        bekle = 60.0 / self.dakikada_istek - (time.time() - self._son)
        if bekle > 0:
            time.sleep(bekle)
        for deneme in range(deneme_sayisi):
            try:
                t0 = time.time()
                metin, girdi, cikti = self._ham_cagri(sistem, mesaj, json_cikti)
                break
            except Exception as hata:
                kota = any(iz in str(hata) for iz in KOTA_IZLERI)
                if not kota and any(iz in str(hata) for iz in KALICI_HATA_IZLERI):
                    raise RuntimeError(f'Düzeltilemeyen API hatası (anahtar, model adı ya da istek biçimi): {str(hata)[:300]}') from hata
                if deneme == deneme_sayisi - 1:
                    raise
                sure = 65 if kota else 10 * (deneme + 1)
                print(f'  {"kota/yoğunluk" if kota else "hata"}: {str(hata)[:150]} -> {sure} sn bekleniyor ({deneme + 1}/{deneme_sayisi})')
                time.sleep(sure)
        self._son = time.time()
        kayit = {'saglayici': self.saglayici, 'model': self.model, 'sicaklik': self.sicaklik,
                 'sistem': sistem, 'mesaj': mesaj, 'metin': metin, 'girdi_token': int(girdi), 'cikti_token': int(cikti),
                 'sure_sn': round(time.time() - t0, 2), 'zaman': time.strftime('%Y-%m-%d %H:%M:%S')}
        if yol:
            with open(yol, 'w', encoding='utf-8') as f:
                json.dump(kayit, f, ensure_ascii=False, indent=1)
        self.istatistik['api_istegi'] += 1
        self.istatistik['girdi_token'] += int(girdi)
        self.istatistik['cikti_token'] += int(cikti)
        return {**kayit, 'onbellekten': False}


def gemini_modeli_sec(api_anahtari, tercih='flash'):
    """Hesapta generateContent destekleyen Gemini modellerini listeler ve kararlı, en yeni 'flash' modelini önerir."""
    from google import genai
    adlar = []
    istemci = genai.Client(api_key=api_anahtari)   # değişkende tutulmalı: geçici istemci silinince bağlantısı kapanır ve liste okunamaz
    for m in istemci.models.list():
        eylemler = getattr(m, 'supported_actions', None) or []
        if 'generateContent' in eylemler:
            adlar.append(m.name.replace('models/', ''))
    istenmeyen = ('preview', 'exp', 'image', 'tts', 'live', 'audio', 'thinking', '8b', 'embedding', 'latest')

    def surum(ad):
        m = re.search(r'gemini-(\d+(?:\.\d+)?)', ad)
        return float(m.group(1)) if m else 0.0
    for tur in ([tercih + '-lite'] if tercih.endswith('lite') else [tercih, tercih + '-lite']):
        adaylar = [a for a in adlar if a.endswith(tur) and not any(x in a for x in istenmeyen)]
        if adaylar:
            return sorted(adaylar, key=surum, reverse=True)[0], adlar
    return None, adlar


def json_ayikla(metin):
    """Model cevabından JSON nesnesini çıkarır (kod bloğu işaretleri vb. temizlenir). Başarısızsa None."""
    if not metin:
        return None
    t = metin.strip()
    t = re.sub(r'^```(?:json)?\s*|\s*```$', '', t)
    try:
        return json.loads(t)
    except Exception:
        a, b = t.find('{'), t.rfind('}')
        if a >= 0 and b > a:
            try:
                return json.loads(t[a:b + 1])
            except Exception:
                return None
    return None


Overwriting /content/drive/MyDrive/bitirme/src/llm_istemci.py


In [3]:
%%writefile /content/drive/MyDrive/bitirme/src/literatur_ajani.py
"""Literatür ajanı (RAG): kanıt paketi oluşturma, sistem istemi ve kaynak doğrulaması.

Ajan görüntüyü görmez; yalnızca metin alır: CNN çıktısı, konsey kapısının gerekçesi, hasta bilgisi,
benzer arşiv vakaları (Adım 4) ve PubMed makaleleri (Adım 6).
"""
import re
import json
import numpy as np
import pandas as pd
import bulanik_kapi as bk
from llm_istemci import json_ayikla

SINIF_TR = {'akiec': 'Aktinik keratoz / Bowen', 'bcc': 'Bazal hücreli karsinom', 'bkl': 'Benign keratoz',
            'df': 'Dermatofibrom', 'mel': 'Melanom', 'nv': 'Melanositik nevüs', 'vasc': 'Vasküler lezyon'}
SINIF_EN = {'akiec': 'actinic keratosis or Bowen disease', 'bcc': 'basal cell carcinoma',
            'bkl': 'seborrheic keratosis or solar lentigo', 'df': 'dermatofibroma', 'mel': 'melanoma',
            'nv': 'melanocytic nevus', 'vasc': 'vascular lesions such as hemangioma or angiokeratoma'}
DOGRULAMA_TR = {'histo': 'patoloji', 'follow_up': 'takip', 'consensus': 'uzman uzlaşısı', 'confocal': 'konfokal mikroskopi'}
CINSIYET_TR = {'male': 'erkek', 'female': 'kadın'}
KOTU_HUYLU = ['mel', 'bcc', 'akiec']
KANIT_YONLERI = ('destekliyor', 'zayiflatiyor', 'belirsiz')

SISTEM_ISTEMI = """Sen bir dermatoloji konseyinde LİTERATÜR AJANISIN. Görevin, bir cilt lezyonu vakası için sana verilen KANIT PAKETİNİ okuyup kaynak gösteren, tarafsız bir kanıt özeti yazmak.

KURALLAR:
1. Yalnızca paketteki bilgileri kullan. Paket dışı bir makale, PMID ya da vaka uydurma; kendi genel bilgini kaynak gibi sunma.
2. Lezyonun görüntüsünü GÖRMÜYORSUN. "Bu lezyonda şu bulgu var/yok" deme. Makalelerdeki dermoskopik bulguları, uzmanın bakması gereken ölçütler olarak yaz.
3. Her iddianın sonuna kaynağını köşeli parantezle yaz: benzer vakalar [V1]...[V5], makaleler [M1]...[M6], CNN çıktısı [CNN], hasta bilgisi [HASTA], konsey kapısı [KAPI], arşiv oylaması [ARSIV]. Kaynaksız iddia yazma.
4. Tarafsız ol. CNN'in kararını savunmak ya da çürütmek senin görevin değil; kanıtın hangi yöne işaret ettiğini dürüstçe bildir. Kanıt zayıf ya da çelişkiliyse bunu açıkça söyle ve "belirsiz" de.
5. Benzer vakaların tanı doğrulama yöntemini dikkate al: patolojiyle doğrulanmış tanı, takip ya da uzman uzlaşısıyla konulan tanıdan daha güçlü kanıttır.
6. Cevabı yalnızca aşağıdaki JSON biçiminde ve Türkçe ver; JSON dışında hiçbir şey yazma:
{"ozet": "3-5 cümlelik kanıt özeti, her cümlede kaynak", "aday_tanilar": [{"sinif": "<kod>", "lehte": "... [Vx][Mx]", "aleyhte": "... [Mx]"}], "en_olasi_tani": "<kod>", "kanit_yonu": "destekliyor | zayiflatiyor | belirsiz", "bakilacak_bulgular": ["... [Mx]"]}

Sınıf kodları: akiec (aktinik keratoz / Bowen), bcc (bazal hücreli karsinom), bkl (benign keratoz), df (dermatofibrom), mel (melanom), nv (melanositik nevüs), vasc (vasküler lezyon).
"kanit_yonu": kanıtın bütün olarak CNN'in kararını destekleyip desteklemediğidir."""


def _softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def _yas(x):
    return 'bilinmiyor' if pd.isna(x) else f'{int(x)}'


class KanitDeposu:
    """Önceki adımların bütün çıktılarını yükler ve tek bir vaka için kanıt paketi üretir."""

    def __init__(self, proje, arama):
        self.arama = arama
        with open(f'{proje}/results/03v2_son_cnn.json', encoding='utf-8') as f:
            son = json.load(f)
        with open(f'{proje}/bulanik/05_kapi_ayarlari.json', encoding='utf-8') as f:
            kapi = json.load(f)
        self.siniflar = son['siniflar']
        log_oncel = np.array(son['log_oncel'])
        meta = pd.read_csv(f'{proje}/splits/ham10000_split.csv')
        self.kume = {k: meta[meta['split'] == k].reset_index(drop=True) for k in ['dogrulama', 'test']}
        self.arsiv = pd.read_csv(f'{proje}/rag/arsiv_meta.csv')
        self.kapi = bk.BulanikKapi(kapi['yuzdelikler'])
        self.esik = kapi['esik']
        self.logit, self.p_cnn, self.karar, self.p_arsiv, self.komsu, self.benzerlik, self.kapi_skor = {}, {}, {}, {}, {}, {}, {}
        kh = [self.siniflar.index(s) for s in KOTU_HUYLU]
        self.kapi_girdi = {}
        for k in self.kume:
            z = np.load(f'{proje}/models/son_cnn_{k}_logit.npy')
            self.logit[k] = z
            self.p_cnn[k] = _softmax(z / son['T'])
            self.karar[k] = (z - son['tau_klinik'] * log_oncel).argmax(1)
            self.p_arsiv[k] = np.load(f'{proje}/rag/arsiv_olasilik_{k}.npy')
            kd = np.load(f'{proje}/rag/komsular_{k}.npz')
            self.komsu[k], self.benzerlik[k] = kd['komsu'], kd['benzerlik']
            self.kapi_skor[k] = np.load(f'{proje}/bulanik/kapi_skoru_{k}.npy')
            n = np.arange(len(z))
            self.kapi_girdi[k] = {'guven': self.p_cnn[k][n, self.karar[k]], 'uyum': self.p_arsiv[k][n, self.karar[k]],
                                  'risk': 0.5 * (self.p_cnn[k][:, kh].sum(1) + self.p_arsiv[k][:, kh].sum(1)),
                                  'yas': self.kume[k]['age'].astype(float).values}

    def konsey_vakalari(self, k):
        return np.where(self.kapi_skor[k] > self.esik)[0]

    def literatur_sorgulari(self, adaylar):
        if len(adaylar) == 1:
            a = SINIF_EN[adaylar[0]]
            return [f'dermoscopic features of {a}', f'lesions mimicking {a} on dermoscopy and differential diagnosis']
        sorgular = [f'dermoscopic criteria to differentiate {SINIF_EN[a]} from {SINIF_EN[b]}'
                    for j, a in enumerate(adaylar) for b in adaylar[j + 1:]]
        return sorgular[:3] + [f'dermoscopic features of {SINIF_EN[adaylar[0]]}']

    def paket(self, k, i, n_komsu=5, n_makale=6, makale_basina=3, ozet_uzunlugu=1200):
        S, v = self.siniflar, self.kume[k].loc[i]
        karar = S[self.karar[k][i]]
        p = self.p_cnn[k][i]
        ilk3 = np.argsort(-p)[:3]
        pa = self.p_arsiv[k][i]
        arsiv3 = np.argsort(-pa)[:3]
        adaylar = list(dict.fromkeys([karar, S[ilk3[0]], S[arsiv3[0]]]))
        # Konsey kapısının gerekçesi
        G = {g: self.kapi_girdi[k][g][i] for g in ['guven', 'uyum', 'risk', 'yas']}
        skor, _, guc = self.kapi.skor(G['guven'], G['uyum'], G['risk'], G['yas'], ayrinti=True)
        kurallar = [(ad, metin, float(guc[ad][0])) for ad, metin, _, _ in bk.KURALLAR if guc[ad][0] > 0.2]
        # Literatür
        sorgular = self.literatur_sorgulari(adaylar)
        makaleler, gorulen = [], set()
        for s in sorgular:
            for _, r in self.arama.ara(s, k=makale_basina).iterrows():
                if r['pmid'] not in gorulen and len(makaleler) < n_makale:
                    gorulen.add(r['pmid'])
                    makaleler.append({'pmid': str(r['pmid']), 'yil': None if pd.isna(r['yil']) else int(r['yil']),
                                      'baslik': r['baslik'], 'ozet': r['ozet'][:ozet_uzunlugu]})
        # Benzer vakalar
        komsular = []
        for j in range(n_komsu):
            a = self.arsiv.loc[self.komsu[k][i, j]]
            komsular.append({'tani': a['dx'], 'dogrulama': DOGRULAMA_TR.get(a['dx_type'], a['dx_type']),
                             'benzerlik': float(self.benzerlik[k][i, j]), 'yas': _yas(a['age']),
                             'cinsiyet': CINSIYET_TR.get(a['sex'], 'bilinmiyor'), 'bolge': a['localization']})
        # Metin
        sat = [f'VAKA [HASTA]: yaş {_yas(v["age"])}, cinsiyet {CINSIYET_TR.get(v["sex"], "bilinmiyor")}, bölge {v["localization"]}.',
               f'CNN [CNN]: klinik eşikli karar = {SINIF_TR[karar]} ({karar}); bu karara kalibre güven %{100 * p[S.index(karar)]:.0f}. '
               'İlk 3 olasılık: ' + ', '.join(f'{SINIF_TR[S[c]]} ({S[c]}) %{100 * p[c]:.0f}' for c in ilk3) + '.',
               f'KONSEY KAPISI [KAPI]: konsey gerekliliği {skor[0]:.1f} (eşik {self.esik:.1f}). Ateşlenen kurallar: '
               + '; '.join(f'{ad} "{metin}" (güç {g:.2f})' for ad, metin, g in kurallar) + '.',
               f'ARŞİV OYLAMASI [ARSIV]: en benzer 50 eğitim vakasının sınıf sıklığına göre düzeltilmiş oyları: '
               + ', '.join(f'{SINIF_TR[S[c]]} ({S[c]}) %{100 * pa[c]:.0f}' for c in arsiv3) + '.',
               '', 'BENZER VAKALAR (eğitim arşivinden, tanısı bilinen):']
        for j, a in enumerate(komsular, 1):
            sat.append(f'[V{j}] {SINIF_TR[a["tani"]]} ({a["tani"]}) | tanı doğrulama: {a["dogrulama"]} | benzerlik {a["benzerlik"]:.2f} | '
                       f'{a["yas"]} yaş, {a["cinsiyet"]}, {a["bolge"]}')
        sat += ['', 'MAKALELER (PubMed özetleri):']
        for j, m in enumerate(makaleler, 1):
            sat.append(f'[M{j}] PMID {m["pmid"]} ({m["yil"]}) {m["baslik"]}\nÖzet: {m["ozet"]}')
        gecerli = {'CNN', 'HASTA', 'KAPI', 'ARSIV'} | {f'V{j}' for j in range(1, len(komsular) + 1)} | {f'M{j}' for j in range(1, len(makaleler) + 1)}
        return {'kume': k, 'indeks': int(i), 'image_id': v['image_id'], 'gercek': v['dx'], 'cnn_karar': karar,
                'arsiv_en_olasi': S[arsiv3[0]], 'adaylar': adaylar, 'kapi_skoru': float(skor[0]), 'sorgular': sorgular,
                'komsular': komsular, 'makaleler': makaleler, 'gecerli_kaynaklar': sorted(gecerli), 'metin': '\n'.join(sat)}


def kaynaklari_bul(metin):
    etiketler = []
    for icerik in re.findall(r'\[([^\[\]]{1,40})\]', metin or ''):
        etiketler += [t for t in re.split(r'[\s,;]+', icerik.strip()) if t]
    return etiketler


def _metinler(cikti):
    parcalar = [cikti.get('ozet', '')]
    for a in cikti.get('aday_tanilar', []) or []:
        if isinstance(a, dict):
            parcalar += [str(a.get('lehte', '')), str(a.get('aleyhte', ''))]
    parcalar += [str(b) for b in (cikti.get('bakilacak_bulgular', []) or [])]
    return [str(p) for p in parcalar if p]


def dogrula(cikti, paket, siniflar):
    """Biçim ve kaynak kontrolü. Uydurma kaynak: pakette olmayan herhangi bir köşeli parantez etiketi."""
    if not isinstance(cikti, dict):
        return {'json_gecerli': False}
    gecerli = set(paket['gecerli_kaynaklar'])
    metinler = _metinler(cikti)
    etiketler = [e for m in metinler for e in kaynaklari_bul(m)]
    uydurma = sorted({e for e in etiketler if e not in gecerli})
    cumleler = [c for c in re.split(r'(?<=[.!?])\s+', str(cikti.get('ozet', '')).strip()) if len(c) > 15]
    kaynaksiz = [c for c in cumleler if not kaynaklari_bul(c)]
    adaylar = [a.get('sinif') for a in (cikti.get('aday_tanilar') or []) if isinstance(a, dict)]
    return {'json_gecerli': True,
            'alanlar_tam': all(a in cikti for a in ['ozet', 'aday_tanilar', 'en_olasi_tani', 'kanit_yonu', 'bakilacak_bulgular']),
            'en_olasi_gecerli': cikti.get('en_olasi_tani') in siniflar,
            'kanit_yonu_gecerli': cikti.get('kanit_yonu') in KANIT_YONLERI,
            'aday_siniflar_gecerli': all(a in siniflar for a in adaylar) and len(adaylar) > 0,
            'kaynak_sayisi': len(etiketler), 'uydurma_kaynaklar': uydurma,
            'kullanilan_makale': len({e for e in etiketler if e.startswith('M') and e in gecerli}),
            'kullanilan_vaka': len({e for e in etiketler if e.startswith('V') and e in gecerli}),
            'ozet_cumle': len(cumleler), 'kaynaksiz_cumle': len(kaynaksiz)}


def _sade(x):
    return str(x or '').strip().lower().translate(str.maketrans('ıİşŞğĞüÜöÖçÇ', 'iissgguuoocc'))


def normallestir(cikti):
    """Küçük biçim farklarını düzeltir: 'Zayıflatıyor' -> 'zayiflatiyor', 'MEL' -> 'mel'."""
    if isinstance(cikti, dict):
        cikti['kanit_yonu'] = _sade(cikti.get('kanit_yonu'))
        cikti['en_olasi_tani'] = _sade(cikti.get('en_olasi_tani'))
        for a in cikti.get('aday_tanilar') or []:
            if isinstance(a, dict):
                a['sinif'] = _sade(a.get('sinif'))
    return cikti


def calistir(paket, istemci, siniflar):
    cevap = istemci.cagir(SISTEM_ISTEMI, paket['metin'], json_cikti=True)
    cikti = normallestir(json_ayikla(cevap['metin']))
    return {'cikti': cikti, 'dogrulama': dogrula(cikti, paket, siniflar), 'ham_metin': cevap['metin'],
            'girdi_token': cevap['girdi_token'], 'cikti_token': cevap['cikti_token'], 'onbellekten': cevap['onbellekten']}


Overwriting /content/drive/MyDrive/bitirme/src/literatur_ajani.py


In [4]:
sys.path.insert(0, f'{PROJE}/src')
import llm_istemci as li, bulanik_kapi as bk, pubmed_arama as pa
importlib.reload(li); importlib.reload(bk); importlib.reload(pa)
import literatur_ajani as la
importlib.reload(la)

if SAGLAYICI == 'gemini' and not MODEL:
    MODEL, tum_modeller = li.gemini_modeli_sec(API_ANAHTARI)
    print('Hesaptaki modeller:', ', '.join(tum_modeller))
    assert MODEL, 'Uygun bir flash modeli bulunamadı: MODEL ayarına yukarıdaki listeden birini elle yaz.'
print(f'Kullanılacak model: {SAGLAYICI} / {MODEL}')

ISTEMCI = li.LLMIstemci(SAGLAYICI, MODEL, API_ANAHTARI, onbellek=f'{PROJE}/ajanlar/onbellek', dakikada_istek=DAKIKADA_ISTEK)
deneme = ISTEMCI.cagir('Yalnızca JSON döndür.', 'Şu JSON nesnesini aynen döndür: {"durum": "tamam"}')
print('Bağlantı testi:', li.json_ayikla(deneme['metin']), '(önbellekten)' if deneme['onbellekten'] else '')

Kullanılacak model: gemini / gemini-3.5-flash
Bağlantı testi: {'durum': 'tamam'} 


## 3. Kanıt deposu
Önceki adımların bütün çıktıları (son CNN logitleri, arşiv komşuları, kapı puanları, PubMed araması) yüklenir. Kontrol: kapı puanları yeniden hesaplanır ve Adım 5'te kaydedilenlerle aynı olmalıdır.

In [5]:
ARAMA = pa.PubmedArama(f'{PROJE}/rag', SORGU_MODELI, CAPRAZ_MODEL)
DEPO = la.KanitDeposu(PROJE, ARAMA)
for k in DEPO.kume:
    yeniden = DEPO.kapi.skor(**DEPO.kapi_girdi[k])
    fark = np.abs(yeniden - DEPO.kapi_skor[k]).max()
    print(f'{k:9s}: {len(DEPO.konsey_vakalari(k))} konsey vakası | kapı puanı Adım 5 ile en büyük fark {fark:.4f} '
          f'-> {"TUTARLI" if fark < 0.01 else "DİKKAT: tutarsız, bana haber ver"}')

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/706k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

... (11 outputs left out)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

dogrulama: 293 konsey vakası | kapı puanı Adım 5 ile en büyük fark 0.0000 -> TUTARLI
test     : 353 konsey vakası | kapı puanı Adım 5 ile en büyük fark 0.0000 -> TUTARLI


## 4. Geliştirme vakaları (yalnızca doğrulama kümesi)
Doğrulamadaki konsey vakalarından 20 tanesi seçilir: 10'unda CNN yanlış, 10'unda doğru. Böylece ajanın hem hatalı hem doğru CNN kararları karşısındaki davranışı görülür. Test kümesine dokunulmaz.

In [6]:
y_dog = DEPO.kume['dogrulama']['dx'].map({s: i for i, s in enumerate(DEPO.siniflar)}).values
konsey = DEPO.konsey_vakalari('dogrulama')
cnn_dogru = DEPO.karar['dogrulama'][konsey] == y_dog[konsey]
n_yanlis = min(N_ORNEK // 2, int((~cnn_dogru).sum()))
SECIM = np.concatenate([rng.choice(konsey[~cnn_dogru], n_yanlis, replace=False),
                        rng.choice(konsey[cnn_dogru], N_ORNEK - n_yanlis, replace=False)])
rng.shuffle(SECIM)
print(f'Seçilen {len(SECIM)} vaka: CNN yanlış {n_yanlis}, CNN doğru {len(SECIM) - n_yanlis}')
print(DEPO.kume['dogrulama'].loc[SECIM, 'dx'].value_counts().to_string())

Seçilen 20 vaka: CNN yanlış 10, CNN doğru 10
dx
nv       8
mel      7
bcc      3
akiec    2


### Ajanın gördüğü kanıt paketi (örnek)
Ajana giden metnin tamamı. Uzun görünebilir; büyük kısmı makale özetleridir.

In [7]:
ornek = DEPO.paket('dogrulama', int(SECIM[0]))
print(ornek['metin'][:6000])
print(f'\n(Toplam {len(ornek["metin"])} karakter | sorgular: {ornek["sorgular"]})')

VAKA [HASTA]: yaş 70, cinsiyet erkek, bölge face.
CNN [CNN]: klinik eşikli karar = Aktinik keratoz / Bowen (akiec); bu karara kalibre güven %23. İlk 3 olasılık: Benign keratoz (bkl) %38, Aktinik keratoz / Bowen (akiec) %23, Bazal hücreli karsinom (bcc) %20.
KONSEY KAPISI [KAPI]: konsey gerekliliği 83.5 (eşik 83.1). Ateşlenen kurallar: K1 "güven DÜŞÜK ise konsey YÜKSEK" (güç 1.00); K2 "uyum DÜŞÜK ise konsey YÜKSEK" (güç 0.90); K12 "yaş İLERİ ve risk YÜKSEK ve DEĞİL(güven YÜKSEK ve uyum YÜKSEK) ise konsey YÜKSEK" (güç 0.69).
ARŞİV OYLAMASI [ARSIV]: en benzer 50 eğitim vakasının sınıf sıklığına göre düzeltilmiş oyları: Bazal hücreli karsinom (bcc) %39, Aktinik keratoz / Bowen (akiec) %34, Benign keratoz (bkl) %16.

BENZER VAKALAR (eğitim arşivinden, tanısı bilinen):
[V1] Aktinik keratoz / Bowen (akiec) | tanı doğrulama: patoloji | benzerlik 0.58 | 75 yaş, erkek, face
[V2] Benign keratoz (bkl) | tanı doğrulama: patoloji | benzerlik 0.56 | 65 yaş, erkek, back
[V3] Bazal hücreli karsinom (bc

## 5. Ajanı çalıştır
Her vaka için paket hazırlanır ve ajana gönderilir. İstekler arasında hız sınırı kadar beklenir. Kota dolarsa hücre bekleyip yeniden dener; tamamen durursa hücreyi sonra yeniden çalıştırman yeterli, biten vakalar önbellekten gelir.

In [8]:
SONUCLAR = []
t0 = time.time()
for j, i in enumerate(SECIM, 1):
    paket = DEPO.paket('dogrulama', int(i))
    s = la.calistir(paket, ISTEMCI, DEPO.siniflar)
    SONUCLAR.append({'paket': paket, **s})
    d, c = s['dogrulama'], (s['cikti'] or {})
    print(f'{j:2d}/{len(SECIM)} {paket["image_id"]} | gerçek {paket["gercek"]:5s} | CNN {paket["cnn_karar"]:5s} | '
          f'arşiv {paket["arsiv_en_olasi"]:5s} | ajan {str(c.get("en_olasi_tani", "?")):5s} | yön {str(c.get("kanit_yonu", "?")):12s} | '
          f'uydurma {len(d.get("uydurma_kaynaklar", []))}{"  (önbellek)" if s["onbellekten"] else ""}')
print(f'\nSüre {(time.time() - t0) / 60:.1f} dk | istatistik: {ISTEMCI.istatistik}')

 1/20 ISIC_0027297 | gerçek bcc   | CNN akiec | arşiv bcc   | ajan akiec | yön belirsiz     | uydurma 0
 2/20 ISIC_0033162 | gerçek nv    | CNN nv    | arşiv mel   | ajan nv    | yön belirsiz     | uydurma 0
 3/20 ISIC_0030581 | gerçek nv    | CNN mel   | arşiv nv    | ajan nv    | yön zayiflatiyor | uydurma 1
 4/20 ISIC_0031908 | gerçek mel   | CNN mel   | arşiv mel   | ajan mel   | yön destekliyor  | uydurma 0
 5/20 ISIC_0033975 | gerçek mel   | CNN df    | arşiv mel   | ajan mel   | yön zayiflatiyor | uydurma 1
 6/20 ISIC_0026268 | gerçek mel   | CNN mel   | arşiv mel   | ajan mel   | yön destekliyor  | uydurma 0
 7/20 ISIC_0030341 | gerçek akiec | CNN bcc   | arşiv bcc   | ajan bcc   | yön destekliyor  | uydurma 0
 8/20 ISIC_0031942 | gerçek nv    | CNN mel   | arşiv mel   | ajan mel   | yön belirsiz     | uydurma 0
 9/20 ISIC_0027365 | gerçek bcc   | CNN bcc   | arşiv bcc   | ajan bcc   | yön belirsiz     | uydurma 0
10/20 ISIC_0028069 | gerçek mel   | CNN mel   | arşiv mel   | aj

## 6. Biçim ve kaynak denetimi
- **JSON geçerli / alanlar tam:** Cevap istenen biçimde mi?
- **Uydurma kaynak:** Pakette olmayan bir etiket kullanıldı mı? Bu, en kritik ölçüt; hedef sıfır.
- **Kaynaksız cümle:** Özet cümlelerinden kaçında hiç kaynak yok?
- **Kullanılan kaynak:** Ajan makaleleri ve benzer vakaları gerçekten kullanıyor mu?

In [9]:
D = pd.DataFrame([{**{'json_gecerli': False, 'alanlar_tam': False, 'uydurma_kaynaklar': [], 'ozet_cumle': 0, 'kaynaksiz_cumle': 0,
                       'kullanilan_makale': 0, 'kullanilan_vaka': 0, 'en_olasi_gecerli': False, 'kanit_yonu_gecerli': False},
                    **r['dogrulama'],
                    'image_id': r['paket']['image_id'], 'gercek': r['paket']['gercek'], 'cnn': r['paket']['cnn_karar'],
                    'arsiv': r['paket']['arsiv_en_olasi'], 'ajan': (r['cikti'] or {}).get('en_olasi_tani'),
                    'yon': (r['cikti'] or {}).get('kanit_yonu'), 'girdi_token': r['girdi_token'], 'cikti_token': r['cikti_token']}
                   for r in SONUCLAR])
DENETIM = {'JSON geçerli': D['json_gecerli'].mean(),
           'Bütün alanlar tam': D['alanlar_tam'].mean(),
           'En olası tanı geçerli kod': D['en_olasi_gecerli'].mean(),
           'Kanıt yönü geçerli değer': D['kanit_yonu_gecerli'].mean(),
           'Uydurma kaynak içeren cevap oranı': (D['uydurma_kaynaklar'].apply(len) > 0).mean(),
           'Toplam uydurma etiket': int(D['uydurma_kaynaklar'].apply(len).sum()),
           'Özette kaynaksız cümle oranı': D['kaynaksiz_cumle'].sum() / max(D['ozet_cumle'].sum(), 1),
           'Cevap başına kullanılan makale (ort.)': D['kullanilan_makale'].mean(),
           'Cevap başına kullanılan benzer vaka (ort.)': D['kullanilan_vaka'].mean()}
for a, b in DENETIM.items():
    print(f'{a:45s}: {b:.3f}' if isinstance(b, float) else f'{a:45s}: {b}')
uyd = D[D['uydurma_kaynaklar'].apply(len) > 0]
if len(uyd):
    print('\nUydurma etiketler:', uyd[['image_id', 'uydurma_kaynaklar']].to_string(index=False))

JSON geçerli                                 : 1.000
Bütün alanlar tam                            : 1.000
En olası tanı geçerli kod                    : 1.000
Kanıt yönü geçerli değer                     : 1.000
Uydurma kaynak içeren cevap oranı            : 0.200
Toplam uydurma etiket                        : 4
Özette kaynaksız cümle oranı                 : 0.012
Cevap başına kullanılan makale (ort.)        : 4.450
Cevap başına kullanılan benzer vaka (ort.)   : 4.650

Uydurma etiketler:     image_id uydurma_kaynaklar
ISIC_0030581           [V1-V5]
ISIC_0033975           [V1-V5]
ISIC_0028069           [V1-V5]
ISIC_0027492           [V1-V5]


## 7. Ön gözlem: ajanın kanıt yönü CNN'in doğruluğuyla ilişkili mi?
**Yalnızca 20 vaka, sonuç çıkarmak için değil, gidişatı görmek için.** Ajan "kanıt CNN'i zayıflatıyor" dediğinde CNN gerçekten daha sık mı yanılıyor? Ajanın en olası tanısı CNN'den ya da arşivden daha mı doğru? Ajan çoğunlukla arşivi mi tekrarlıyor?

In [10]:
D['cnn_dogru'] = D['cnn'] == D['gercek']
print('Kanıt yönüne göre CNN doğruluğu (vaka sayısı ve oran):')
print(D.groupby('yon')['cnn_dogru'].agg(['count', 'mean']).round(2).to_string())
print(f"\nDoğruluk (bu 20 vakada): CNN {D['cnn_dogru'].mean():.2f} | arşiv {(D['arsiv'] == D['gercek']).mean():.2f} | "
      f"ajan {(D['ajan'] == D['gercek']).mean():.2f}")
print(f"Ajanın en olası tanısı arşivle aynı: %{100 * (D['ajan'] == D['arsiv']).mean():.0f} | CNN ile aynı: %{100 * (D['ajan'] == D['cnn']).mean():.0f}")

Kanıt yönüne göre CNN doğruluğu (vaka sayısı ve oran):
              count  mean
yon                      
belirsiz         10  0.60
destekliyor       6  0.67
zayiflatiyor      4  0.00

Doğruluk (bu 20 vakada): CNN 0.50 | arşiv 0.65 | ajan 0.65
Ajanın en olası tanısı arşivle aynı: %70 | CNN ile aynı: %75


## 8. Örnek cevaplar (gözle inceleme)
Bir CNN hatası ve bir doğru CNN kararı için ajanın cevabı. Kaynak olarak gösterilen makalelerin başlıkları da yazılır; böylece **kaynağın iddiayı gerçekten destekleyip desteklemediğini** gözle kontrol edebiliriz. Otomatik denetim bunu yapamaz.

In [11]:
def goster(r):
    p, c = r['paket'], (r['cikti'] or {})
    print('=' * 110)
    print(f'{p["image_id"]} | gerçek: {p["gercek"]} | CNN: {p["cnn_karar"]} | arşiv: {p["arsiv_en_olasi"]} | aday tanılar: {p["adaylar"]}')
    print('Benzer vakalar:', ' | '.join(f'V{j}: {v["tani"]} ({v["dogrulama"]}, {v["benzerlik"]:.2f})' for j, v in enumerate(p['komsular'], 1)))
    print(f'\nÖZET: {c.get("ozet")}')
    for a in c.get('aday_tanilar', []) or []:
        print(f'  - {a.get("sinif")}: LEHTE {a.get("lehte")} | ALEYHTE {a.get("aleyhte")}')
    print(f'EN OLASI: {c.get("en_olasi_tani")} | KANIT YÖNÜ: {c.get("kanit_yonu")}')
    for b in c.get('bakilacak_bulgular', []) or []:
        print(f'  * {b}')
    atif = sorted({e for m in la._metinler(c) for e in la.kaynaklari_bul(m) if e.startswith('M')})
    print('\nGösterilen makaleler:')
    for e in atif:
        j = int(e[1:]) - 1
        if 0 <= j < len(p['makaleler']):
            print(f'  [{e}] PMID {p["makaleler"][j]["pmid"]}: {p["makaleler"][j]["baslik"][:120]}')
    print('Denetim:', {k: v for k, v in r['dogrulama'].items() if k in ['uydurma_kaynaklar', 'kaynaksiz_cumle', 'ozet_cumle']})

yanlislar = [r for r in SONUCLAR if r['paket']['cnn_karar'] != r['paket']['gercek']]
dogrular = [r for r in SONUCLAR if r['paket']['cnn_karar'] == r['paket']['gercek']]
for r in yanlislar[:1] + dogrular[:1]:
    goster(r)

ISIC_0027297 | gerçek: bcc | CNN: akiec | arşiv: bcc | aday tanılar: ['akiec', 'bkl', 'bcc']
Benzer vakalar: V1: akiec (patoloji, 0.58) | V2: bkl (patoloji, 0.56) | V3: bcc (patoloji, 0.55) | V4: mel (patoloji, 0.54) | V5: bkl (patoloji, 0.54)

ÖZET: Hastanın 70 yaşında erkek olduğu ve lezyonun yüz bölgesinde yer aldığı bildirilmiştir [HASTA]. CNN klinik kararı %23 güvenle akiec yönünde olmasına rağmen, ham olasılıklarda %38 ile bkl ilk sıradadır [CNN]. Düşük güven ve uyumsuzluk nedeniyle konsey kapısı tetiklenmiştir [KAPI]. Arşiv oylamasında bcc %39 ile ilk sırada yer alırken, en yüksek benzerliğe sahip vaka patoloji ile doğrulanmış bir akiec vakasıdır [ARSIV, V1]. Dermoskopik literatür keratinosit kaynaklı lezyonların ve pigmente yapıların ayırıcı tanısı için çeşitli kriterler tanımlamaktadır [M4, M5].
  - akiec: LEHTE CNN klinik eşikli kararı akiec sınıfıdır [CNN]. Patoloji ile doğrulanmış 0.58 benzerlikteki en yakın vaka yüz bölgesinde akiec tanısı almıştır [V1]. Arşiv oylamasında 

## 9. Maliyet tahmini (büyük deney için)
Ölçülen token sayılarıyla, Adım 11'deki büyük deneyin (doğrulama + test konsey vakaları × 3 ajan) yaklaşık token ihtiyacı hesaplanır. Fiyat bilgisi sağlayıcının güncel fiyat sayfasından alınıp aşağıya yazılırsa dolar karşılığı da gösterilir.

In [12]:
FIYAT_GIRDI = None    # $ / 1 milyon girdi tokenı (ör. 0.10)
FIYAT_CIKTI = None    # $ / 1 milyon çıktı tokenı (ör. 0.40)
ort_girdi, ort_cikti = D['girdi_token'].mean(), D['cikti_token'].mean()
n_konsey = sum(len(DEPO.konsey_vakalari(k)) for k in DEPO.kume)
cagri = n_konsey * 3
print(f'Çağrı başına ortalama: {ort_girdi:.0f} girdi, {ort_cikti:.0f} çıktı tokenı')
print(f'Büyük deney: {n_konsey} konsey vakası × 3 ajan ≈ {cagri} çağrı ≈ {cagri * ort_girdi / 1e6:.2f} milyon girdi, '
      f'{cagri * ort_cikti / 1e6:.2f} milyon çıktı tokenı')
if FIYAT_GIRDI is not None and FIYAT_CIKTI is not None:
    print(f'Tahmini maliyet: ${cagri * (ort_girdi * FIYAT_GIRDI + ort_cikti * FIYAT_CIKTI) / 1e6:.2f}')

Çağrı başına ortalama: 2142 girdi, 2203 çıktı tokenı
Büyük deney: 646 konsey vakası × 3 ajan ≈ 1938 çağrı ≈ 4.15 milyon girdi, 4.27 milyon çıktı tokenı


## 10. Kaydet

In [13]:
with open(f'{PROJE}/ajanlar/{ON}_literatur_gelistirme.jsonl', 'w', encoding='utf-8') as f:
    for r in SONUCLAR:
        f.write(json.dumps(r, ensure_ascii=False, default=str) + '\n')
ozet = {'saglayici': SAGLAYICI, 'model': MODEL, 'sicaklik': ISTEMCI.sicaklik, 'n_vaka': len(SONUCLAR), 'kume': 'dogrulama',
        'sistem_istemi': la.SISTEM_ISTEMI, 'denetim': {k: float(v) for k, v in DENETIM.items()},
        'token': {'girdi_ort': float(ort_girdi), 'cikti_ort': float(ort_cikti), 'tahmini_buyuk_deney_cagri': int(cagri)},
        'vakalar': [{'image_id': r['paket']['image_id'], 'gercek': r['paket']['gercek'], 'cnn': r['paket']['cnn_karar'],
                     'arsiv': r['paket']['arsiv_en_olasi'], 'sorgular': r['paket']['sorgular'],
                     'makale_pmid': [m['pmid'] for m in r['paket']['makaleler']],
                     'cikti': r['cikti'], 'dogrulama': r['dogrulama']} for r in SONUCLAR]}
with open(f'{PROJE}/results/{ON}_literatur_ajani_gelistirme.json', 'w', encoding='utf-8') as f:
    json.dump(ozet, f, ensure_ascii=False, indent=2, default=float)
print('Kaydedildi:')
for p in ['src/llm_istemci.py', 'src/literatur_ajani.py', f'ajanlar/{ON}_literatur_gelistirme.jsonl', f'results/{ON}_literatur_ajani_gelistirme.json']:
    print(' ', p, '✓' if os.path.exists(f'{PROJE}/{p}') else 'YOK!')

Kaydedildi:
  src/llm_istemci.py ✓
  src/literatur_ajani.py ✓
  ajanlar/07_literatur_gelistirme.jsonl ✓
  results/07_literatur_ajani_gelistirme.json ✓
